In [ ]:
# Working Scraper for 1999-2024
import requests
from bs4 import BeautifulSoup
import csv
import os
import pandas as pd
import re
import unicodedata
import regex

# Set working directory and output directory
input_csv_path = 
output_dir = 

# Make sure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Read the URLs from the CSV file
urls = pd.read_csv(input_csv_path)['URL'].tolist()  # Assuming the CSV has a column named 'URL'

# Function to normalize and clean text, preserving accents
def normalize_text(text):
    text = unicodedata.normalize('NFKC', text)
    text = regex.sub(r'\p{Mn}+', '', text)
    return text

# Function to add unique institutions, avoiding redundancy
def add_institution(institution_list, institution):
    if institution not in institution_list and not any(institution in item for item in institution_list):
        institution_list.append(institution)
    return institution_list

# Function to extract author from text
def extract_author(text):
    match = re.search(r'Question no.*?by\s+(.*)', text, re.IGNORECASE)
    
    if match:
        
        return match.group(1).strip()
    return None
# Updated function to handle "rapporteur", "author", "Ombudsman", "draft", and "chairman" detection
def clean_quote_and_identify_roles(quote, other_roles, italic_text):
    quote = quote.strip()

    # Detect and remove "rapporteur", "author", "Ombudsman", and "chairman"
    for role in ['rapporteur', 'author', 'Ombudsman', 'chairman']:
        if re.search(rf'\b{role}\b[.,;]?', italic_text, flags=re.IGNORECASE):
            # If chairman is found, move the entire italic_text to other_roles
            if 'chairman' in role.lower():
                other_roles = italic_text.strip() if not other_roles else f"{other_roles}, {italic_text.strip()}"
                quote = re.sub(re.escape(italic_text), '', quote, flags=re.IGNORECASE).strip()
            else:
                other_roles = role.capitalize() if not other_roles else f"{other_roles}, {role.capitalize()}"
                quote = re.sub(rf'\b{role}\b[.,;]?', '', quote, flags=re.IGNORECASE).strip()

    # Detect the root "draft" and add the full italic text to other_roles
    if re.search(r'\b\w*draft\w*\b', italic_text, flags=re.IGNORECASE):
        other_roles = italic_text.strip() if not other_roles else f"{other_roles}, {italic_text.strip()}"
        quote = re.sub(re.escape(italic_text), '', quote, flags=re.IGNORECASE).strip()

    return quote, other_roles

# Function to process annex URLs
def process_annex_url(url):
    print(f"Processing {url} (annex format)...")
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')

    subjects = []
    authors = []
    questions = []
    answers = []

    # Extract all second-level nested tables (tables within tables)
    tables = soup.find_all('table')

    for table in tables:
        if table.find_parent('table'):  # Only process if the table is nested within another table
            # Initialize variables
            subject = None
            author = None
            question = ""
            answer = ""
            in_answer_section = False

            # Find the "Subject:" text in each table and the corresponding author
            for row in table.find_all('tr'):
                row_text = normalize_text(row.get_text(separator=" ").strip())
                if row_text.lower().startswith("subject:"):
                    subject = row_text[len("Subject:"):].strip()  # Extract text after "Subject:"
                    
                # Look for the "Question no" text to find the author
                if "question no" in row_text.lower():
                    author = extract_author(row_text)
                    
            # Extract the question from <p> tags until a paragraph starts with a parenthesis
            paragraphs = table.find_all('p')
            for paragraph in paragraphs:
                paragraph_text = normalize_text(paragraph.get_text().strip())
                if paragraph_text.startswith("("):
                    in_answer_section = True

                if in_answer_section:
                    answer += paragraph_text + " "
                else:
                    question += paragraph_text + " "

            # Clean up question and answer by stripping any extra whitespace
            question = question.strip()
            answer = answer.strip()

            # Add to the list if a subject, author, question, and answer are found
            if subject or author or question or answer:
                subjects.append(subject)
                authors.append(author)
                questions.append(question)
                answers.append(answer)

    if subjects:
        output_file_path = os.path.join(output_dir, f"{url.split('/')[-1].replace('.html', '')}_annex_format.csv")
        with open(output_file_path, 'w', encoding='utf-8', newline='') as csv_file:
            fieldnames = ['Subject', 'Author', 'Question', 'Answer']
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()

            for subject, author, question, answer in zip(subjects, authors, questions, answers):
                writer.writerow({
                    'Subject': subject,
                    'Author': author,
                    'Question': question,
                    'Answer': answer
                })

        print(f"Results saved to {output_file_path}")
    else:
        print(f"No data found for {url}.")
# Modified function to process URLs
def process_url(url):
    # Logic to determine if the URL is an annex link
    if "ANN-" in url:  # Adjust this condition based on your criteria for identifying annex links
        process_annex_url(url)
        return

    print(f"Processing {url}...")
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')

    speakers = []
    last_speaker_added = None

    tables = [table for table in soup.find_all('table') 
              if table.find_parent('table') and table.find_parent('table').find_parent('table')]

    for table in tables:
        node_name = table.find('span', class_='bold')
        italics = table.find_all('span', class_='italic')

        # If no bold text is found, skip this table
        if not node_name:
            continue

        speaker_text = normalize_text(node_name.text.strip())

        # Skip processing if the text contains the words "report" or "draft"
        if any(word in speaker_text.lower() for word in ["report", "draft", "motion"]):
            continue

        if speaker_text.lower().startswith("in the chair"):
            continue

        # Initialize variables to avoid UnboundLocalError
        node_party_parentheses = None
        node_country_parentheses = None
        form_of_speech = "Spoken"  # Default to "Spoken"

        party_match = re.search(r'\((.*?)\)', speaker_text)
        if party_match:
            node_party_parentheses = party_match.group(1)
            speaker = re.sub(r'\(.*?\)', '', speaker_text).strip()
        else:
            speaker = speaker_text

        node_institutions = []
        node_other_roles = None

        # Enhanced parsing to handle multiple elements in a single italic span
        for italic in italics:
            italic_text = normalize_text(italic.text)

            # Check for country information
            country_match = re.search(r'\((.*?)\)', italic_text)
            if country_match:
                potential_country = country_match.group(1).strip()
                # Only add to country if it's a single word
                if potential_country.count(' ') == 0 and potential_country.lower() not in ['applause', 'inaudible', 'heckling', 'parliament agreed to accept the oral amendment', 'the house accorded the speaker a standing ovation.', 'loud applause', 'laughter', 'the president cut off the speaker', 'applause from the left']:
                    node_country_parentheses = potential_country
                italic_text = re.sub(r'\(.*?\)', '', italic_text)  # Remove country info from this span

            # Check for "in writing" or similar form of speech indicators
            if re.search(r'\bin writing\b', italic_text, re.IGNORECASE):
                form_of_speech = "In writing"
                italic_text = re.sub(r'\bin writing\b.*?(?=[.,;:]|\s|$)', '', italic_text, flags=re.IGNORECASE).strip()

            # Check for various roles and institutions
            if 'vice-president of the commission' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'Vice-President of the Commission')
            elif 'vice-president of the european commission' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'Vice-President of the European Commission')
            elif 'member of the commission' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'Member of the Commission')
            elif 'commission' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'Commission')
                
            if 'president-in-office of the council' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'President-in-Office of the Council')
            elif 'member of the council' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'Member of the Council')
            elif 'council' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'Council')
            if 'president of the european investment bank' in italic_text.lower():
                node_institutions = add_institution(node_institutions, 'President of the European Investment Bank')

            # Use the enhanced function
            quote_cleaned, node_other_roles = clean_quote_and_identify_roles(italic_text, node_other_roles, italic_text)

        node_group = None
        for italic in italics:
            italic_text = normalize_text(italic.text)
            group_match = re.search(r'(in the name of|on behalf of).*?group', italic_text, re.IGNORECASE)
            if group_match:
                node_group = group_match.group(0).strip()
                break

        if node_name:
            party = node_party_parentheses if node_party_parentheses else None
            country = node_country_parentheses if node_country_parentheses else None
            group = node_group if node_group else None

            # Check if the speaker is the President and set country, other_roles, and institution to None if true
            if "president" in speaker.lower():
                country = None
                node_other_roles = None  # Clear other roles
                node_institutions = []  # Clear institutions

            institution = ', '.join(node_institutions) if node_institutions else None
            other_roles = node_other_roles if node_other_roles else None

            node_quotes = table.find_all('p', class_='contents')
            full_quote = ' '.join(normalize_text(p.text.strip()) for p in node_quotes)

            # Skip if the quote contains "oral explanations of vote" or "written explanations of vote"
            if 'oral explanations of vote' in full_quote.lower() or 'written explanations of vote' in full_quote.lower():
                continue

            # Remove the text that's already in other_roles from the full quote
            if other_roles:
                for role in other_roles.split(', '):
                    full_quote = re.sub(re.escape(role), '', full_quote, flags=re.IGNORECASE).strip()

            if institution:
                full_quote = full_quote.replace(institution, '').strip()

            # Remove "in writing" from the quote if it's there
            if form_of_speech == "In writing":
                full_quote = re.sub(r'\b(in writing)\b', '', full_quote, flags=re.IGNORECASE).strip()

            # Special handling for group names with overlapping country names
            if group:
                group_text_in_quote = re.escape(group.replace('-', '[-\s]*'))
                group_pattern = rf"(in the name of|on behalf of)[^\S\r\n]*{group_text_in_quote}[.,;]?\s*group"
                full_quote = re.sub(group_pattern, '', full_quote, flags=re.IGNORECASE).strip()

            unique_key = (speaker, party)

            if speaker != last_speaker_added:
                last_speaker_added = speaker

                quote_cleaned = full_quote.replace(speaker, '').replace(f'({party})' if party else '', '').replace(f'({country})' if country else '', '').strip()

                quote_cleaned = re.sub(re.escape(speaker), '', quote_cleaned).strip()
                if party:
                    quote_cleaned = re.sub(re.escape(party), '', quote_cleaned).strip()

                # Conditional to skip removing "DE" if it's part of the PPE-DE Group and ensure country is not None
                if country and not (country == "DE" and "on behalf of the PPE-DE Group" in (group or "")):
                    quote_cleaned = re.sub(re.escape(country), '', quote_cleaned).strip()

                if group:
                    quote_cleaned = re.sub(re.escape(group), '', quote_cleaned).strip()
                if institution:
                    quote_cleaned = re.sub(re.escape(institution), '', quote_cleaned).strip()

                speaker_info = {
                    'Speaker': speaker, 
                    'Party': party, 
                    'Country': country, 
                    'Group': group,  
                    'Other institution': institution, 
                    'Other Roles': other_roles,  
                    'Form of Speech': form_of_speech,  # New column added
                    'Quote': quote_cleaned  # Quote column
                }
                speakers.append(speaker_info)

    if speakers:
        output_file_path = os.path.join(output_dir, f"{url.split('/')[-1].replace('.html', '')}.csv")
        with open(output_file_path, 'w', encoding='utf-8', newline='') as csv_file:
            fieldnames = ['Speaker', 'Party', 'Country', 'Group', 'Other institution', 'Other Roles', 'Form of Speech', 'Quote']  # Adjust field order
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()

            for speaker in speakers:
                writer.writerow(speaker)

        print(f"Results saved to {output_file_path}")
    else:
        print(f"No speakers found for {url}.")

# Process all URLs
for url in urls:
    process_url(url)

print("All URLs processed.")


